In [1]:
## 加载包
from bigdatasource.api import DataSource
from bigdata.api.datareader import D
from biglearning.api import M
from biglearning.api import tools as T
from biglearning.module2.common.data import Outputs

import pandas as pd
import numpy as np
import math
import warnings
import datetime
import dai

from zipline.finance.commission import PerOrder
from zipline.api import get_open_orders
from zipline.api import symbol

from bigtrader.sdk import *
from bigtrader.utils.my_collections import NumPyDeque
from bigtrader.constant import OrderType
from bigtrader.constant import Direction

In [2]:
## 提因子及运算用SQL['alpha_research_1', 'alpha_research_7', 'alpha_fa_37', 'alpha_fa_49', 'alpha_research_31']
sql = """
pragma enable_pushdown_window;
WITH table1 AS(
SELECT A.date, A.instrument, A.rps_lf / B.close AS factor1 FROM cn_stock_factors_financial A JOIN cn_stock_bar1d B ON A.date = B.date AND A.instrument = B.instrument                                WHERE B.date >= '2015-01-01' ORDER BY B.date, B.instrument),
table2 AS(
SELECT date, instrument, m_avg(amount, 240) AS factor2 FROM cn_stock_bar1d WHERE date >= '2015-01-01' ORDER BY date, instrument),
table3 AS(
SELECT date, instrument, asset_impairment_loss_ttm AS factor3 FROM cn_stock_financial_pit_ttm WHERE date >= '2015-01-01' ORDER BY date),
table4 AS(
SELECT A.instrument, A.date, (B.total_market_cap/A.np_atoopc_ly) AS factor4 FROM cn_stock_financial_pit_ly AS A JOIN cn_stock_valuation AS B ON A.instrument = B.instrument AND A.date = B.date WHERE A.date >= '2015-01-01' ORDER BY A.instrument, A.date),
table5 AS(
SELECT date, instrument, m_avg(turn, 60) AS factor5 FROM cn_stock_bar1d WHERE date >= '2015-01-01' ORDER BY date, instrument),
table_0 AS (
-- 合并所有表
SELECT *
FROM table1
JOIN table2 ON table1.date = table2.date AND table1.instrument = table2.instrument
JOIN table3 ON table2.date = table3.date AND table2.instrument = table3.instrument
JOIN table4 ON table3.date = table4.date AND table3.instrument = table4.instrument
JOIN table5 ON table4.date = table5.date AND table4.instrument = table5.instrument ORDER BY table1.date, table1.instrument)
-- 给因子值做排名, 再加总各个排名值得到y因子
SELECT date, instrument,
   pct_rank_by(date, 1*factor1) AS factor_revised_1,
   pct_rank_by(date, -1*factor2) AS factor_revised_2,
   pct_rank_by(date, 1*factor3) AS factor_revised_3,
   pct_rank_by(date, -1*factor4) AS factor_revised_4,
   pct_rank_by(date, -1*factor5) AS factor_revised_5,
   factor_revised_1 + factor_revised_2 + factor_revised_3 + factor_revised_4 + factor_revised_5 AS y
FROM table_0
ORDER BY date, y DESC;
"""
print(sql)

In [3]:
## 设置开始和结束时间
start_date = "2020-01-01"
end_date = "2023-10-30"

data = dai.query(sql).df()
instruments = {'market': 'CN_STOCK_A', 'instruments': list(data.instrument.unique()), 'start_date': start_date, 'end_date': end_date}
instruments = DataSource.write_pickle(instruments)

In [4]:
ds = DataSource.write_df(data)
ds.read_df()

In [5]:
stock_num = 10
hold_day = 5

# 回测引擎：初始化函数，只执行一次
def m_initialize_bigquant_run(context):

    # 加载预测数据
    context.test_data = context.options['data'].read_df()
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    # 设置买入的股票数量，这里买入预测股票列表排名靠前的10只
    context.stock_count = stock_num
    # 每只股票等权权重
    context.stock_weights = 1/context.stock_count
    # 持仓周期
    context.hold_days = hold_day

# 回测引擎：每日数据处理函数，每天执行一次
def m_handle_data_bigquant_run(context, data):
    
    if context.trading_day_index % context.hold_days != 0:
        return 
    
    today = data.current_dt.strftime('%Y-%m-%d')
    # 获取当前持仓
    positions = {e: p.amount for e, p in context.portfolio.positions.items()}
    # 按日期过滤得到今日数据
    today_data = context.test_data[context.test_data.date == today] 
    
    # 今日需要买入的股票
    stocks_buy = today_data.instrument.iloc[0:context.stock_count].to_list()
    #stocks_buy_name = today_data.name.iloc[0:context.stock_count].to_list() #打印显示用
    #print(today,"买入股票池：",stocks_buy,'\n',stocks_buy_name)
    # 卖出
    for instrument in positions.keys():
        if instrument not in stocks_buy:
            context.order_target(context.symbol(instrument), 0)
            #print(today,"卖出",instrument)
    # 买入
    cash_per_instrument = context.portfolio.portfolio_value * context.stock_weights
    #print(today, cash_per_instrument)
    for instrument in stocks_buy:
        if instrument not in positions.keys():
            context.order_value(context.symbol(instrument), cash_per_instrument)
            #print(today,"买入",instrument)          

# 回测引擎：准备数据，只执行一次
def m_prepare_bigquant_run(context):
    pass

# 交易引擎：每个单位时间开盘前调用一次。
def m_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# 交易引擎：tick数据处理函数，每个tick执行一次
def m_handle_tick_bigquant_run(context, tick):
    pass

# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m_handle_trade_bigquant_run(context, trade):
    pass

# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m_handle_order_bigquant_run(context, order):
    pass

# 交易引擎：盘后处理函数，每日盘后执行一次
def m_after_trading_bigquant_run(context, data):
    pass

m = M.hftrade.v2(
    instruments=instruments,
    options_data=ds,
    start_date='',
    end_date='',
    initialize=m_initialize_bigquant_run,
    before_trading_start=m_before_trading_start_bigquant_run,
    handle_tick=m_handle_tick_bigquant_run,
    handle_data=m_handle_data_bigquant_run,
    handle_trade=m_handle_trade_bigquant_run,
    handle_order=m_handle_order_bigquant_run,
    after_trading=m_after_trading_bigquant_run,
    capital_base=1000000.1248,
    frequency='daily',
    price_type='真实价格',
    product_type='股票',
    before_start_days='0',
    volume_limit=1,
    order_price_field_buy='open',
    order_price_field_sell='open',
    benchmark='000300.SH',
    plot_charts=True,
    disable_cache=False,
    replay_bdb=False,
    show_debug_info=False,
    backtest_only=False
)